In [ ]:
# Meth3D-Net V6 — Pseudo-Clinical Risk Model
# Corrected notebook: all bugs fixed, figure saving added.
# Compatible with Kaggle / local environments.

# Standard Kaggle environment info
import os
print("Python environment ready.")


## Step 1: Install Dependencies and Import Libraries

In [ ]:
!pip install lifelines xgboost shap --quiet

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for Kaggle & local
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lifelines import KaplanMeierFitter, CoxPHFitter
import shap

# ── Output directory for figures ─────────────────────────────────────────────
os.makedirs("figures", exist_ok=True)
print("All libraries imported. Figures will be saved to ./figures/")


## Step 2: Locate Data Files

In [ ]:
def find_file(filename, search_dirs=None):
    """Search multiple directories for a file."""
    if search_dirs is None:
        search_dirs = ["/kaggle/input", ".", "/mnt/user-data/uploads"]
    for base in search_dirs:
        for root, dirs, files in os.walk(base):
            if filename in files:
                return os.path.join(root, filename)
    return None

files_needed = [
    "Final_Methylation_Scores.csv",
    "Final_DNAmAge_Robust.csv",
    "Onco_TSG_Methylation_Table.csv",
    "lncRNA_candidates.csv",
]

paths = {}
for f in files_needed:
    p = find_file(f)
    paths[f] = p
    status = "✅ found" if p else "❌ NOT FOUND"
    print(f"{status}: {f}  →  {p}")


## Step 3: Load Primary Dataset

In [ ]:
# Load the primary methylation scores file
data_path = paths.get("Final_Methylation_Scores.csv")
if data_path is None:
    raise FileNotFoundError(
        "Final_Methylation_Scores.csv not found. "
        "Upload it to Kaggle or place it in the working directory."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {df.shape[0]} samples × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
print(df.describe())


## Step 4: Feature Engineering

In [ ]:
# Derived interaction features
df["Onco_TSG_ratio"]    = df["OncoScore"] / (df["TSGScore"].abs() + 1e-6)
df["Age_Instability"]   = df["DNAmAge"] * df["EpigeneticInstability"]
df["Acceleration_binary"] = df["IsAccelerated"].astype(int)

print("Feature engineering complete.")
print(df[["Onco_TSG_ratio","Age_Instability","Acceleration_binary"]].describe())


## Step 5: Define Risk Groups

**Bug fixed:** The original notebook used `CancerScore > 0.7` as the threshold.
However, CancerScore in this dataset ranges from ≈ −0.60 to +0.30, so that
threshold yields **zero** high-risk samples. The correct threshold is the
**75th percentile**, which gives a balanced and interpretable high-risk group.


In [ ]:
# FIX: CancerScore in this dataset ranges ~−0.60 to +0.30.
# Original threshold of > 0.7 gave 0 positive samples (bug).
# Correct approach: 75th percentile split for high-risk group.

threshold = df["CancerScore"].quantile(0.75)
df["risk_group"] = (df["CancerScore"] > threshold).astype(int)  # 1 = high risk

print(f"Risk threshold (75th pct): {threshold:.4f}")
print("Risk group distribution:")
print(df["risk_group"].value_counts())
print(df["risk_group"].value_counts(normalize=True).round(3))


## Step 6: Define Features and Split Data

In [ ]:
features = [
    "DNAmAge",
    "AgeAcceleration",
    "EpigeneticInstability",
    "IsAccelerated",          # bool → int automatically handled by sklearn
]

X = df[features].copy()
X["IsAccelerated"] = X["IsAccelerated"].astype(int)   # explicit cast
y = df["risk_group"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Class balance (train):\n{y_train.value_counts()}")


## Step 7: Random Forest Classifier (Imbalance-Corrected)

In [ ]:
# FIX: roc_auc_score import was missing in the original notebook at this point.
# It is now imported at the top of the notebook (Step 1).

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight="balanced",   # handles class imbalance
    random_state=42
)

model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, y_prob)
print(f"Random Forest AUC: {rf_auc:.4f}")


## Step 8: 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")

print(f"CV AUC (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold scores: {[round(s,4) for s in cv_scores]}")


## Step 9: Feature Importance — Random Forest

**Figure S11** (Supplementary Figure)


In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2E5FA3" if v < importance.max() else "#C0392B" for v in importance.values]
importance.plot(kind="barh", ax=ax, color=colors, edgecolor="white")

ax.set_title("Figure S11. Feature Importance — Random Forest Cancer Risk Model",
             fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("Mean Decrease in Impurity (Gini Importance)", fontsize=10)
ax.set_ylabel("Epigenetic Feature", fontsize=10)
ax.spines[["top","right"]].set_visible(False)
ax.axvline(importance.mean(), color="grey", linestyle="--", linewidth=0.8, label="Mean importance")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("figures/FigS11_RF_Feature_Importance.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: figures/FigS11_RF_Feature_Importance.png")


## Step 10: XGBoost Classifier

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss",
    verbosity=0
)

xgb.fit(X_train, y_train)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, y_prob_xgb)
print(f"XGBoost AUC: {xgb_auc:.4f}")


## Step 11: ROC Curve Comparison — RF vs XGBoost

**Figure S12** (Supplementary Figure)


In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(6, 5))

for name, probs in [("Random Forest", y_prob), ("XGBoost", y_prob_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0,1],[0,1],"k--", linewidth=0.8, label="Random classifier")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("Figure S12. ROC Curve — Cancer Risk Classifiers\n(Random Forest vs XGBoost)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/FigS12_ROC_Comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: figures/FigS12_ROC_Comparison.png")


## Step 12: SHAP Feature Interpretation — XGBoost

**Figure S13** (Supplementary Figure)


In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
shap.summary_plot(shap_values, X_test, show=False, plot_type="dot")

ax = plt.gca()
ax.set_title("Figure S13. SHAP Summary Plot — XGBoost Cancer Risk Model\n"
             "(Feature impact on high-risk prediction)",
             fontsize=11, fontweight="bold", pad=10)

plt.tight_layout()
plt.savefig("figures/FigS13_SHAP_Summary.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: figures/FigS13_SHAP_Summary.png")


## Step 13: Pseudo-Survival Analysis Setup

DNAmAge is used as a proxy for disease progression time.
Epigenetic age acceleration (`IsAccelerated`) is used as the event indicator.


In [ ]:
# Pseudo-survival columns
# FIX: cell 36 in original notebook redefined risk_group using median split,
# then cells 39 correctly set time/event, but cell 41 tried to print cox_df
# before it was defined (bug). Merged and reordered here correctly.

df["time"]  = df["DNAmAge"]              # progression proxy (years)
df["event"] = df["IsAccelerated"].astype(int)   # 1 = accelerated ageing event

# Reaffirm risk group with median split for survival analysis
df["risk_group"] = (df["CancerScore"] > df["CancerScore"].median()).astype(int)

print("Pseudo-survival setup:")
print(f"  Time range:  {df['time'].min():.1f} – {df['time'].max():.1f} years")
print(f"  Events (n):  {df['event'].sum()} of {len(df)} ({df['event'].mean()*100:.1f}%)")
print(f"  High risk:   {df['risk_group'].sum()} samples")


## Step 14: Kaplan–Meier Pseudo-Survival Curves

**Figure S14** (Supplementary Figure)


In [ ]:
kmf = KaplanMeierFitter()

fig, ax = plt.subplots(figsize=(7, 5))

colors = {0: "#2E5FA3", 1: "#C0392B"}
labels = {0: "Low Risk", 1: "High Risk"}

for label, group in df.groupby("risk_group"):
    kmf.fit(group["time"], group["event"],
            label=f"{labels[label]} (n={len(group)})")
    kmf.plot_survival_function(ax=ax, ci_show=True, color=colors[label], linewidth=2)

ax.set_title("Figure S14. Pseudo-Survival by Epigenetic Cancer Risk Group\n"
             "(DNAmAge as progression proxy; event = accelerated epigenetic ageing)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Epigenetic Age (Horvath DNAmAge, years)", fontsize=11)
ax.set_ylabel("Pseudo-Survival Probability", fontsize=11)
ax.spines[["top","right"]].set_visible(False)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("figures/FigS14_KaplanMeier.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: figures/FigS14_KaplanMeier.png")


## Step 15: Cox Proportional Hazards Model

In [ ]:
# FIX: Original notebook had print(cox_df.columns) BEFORE cox_df was defined (cell 41 vs 42).
# cox_df is now defined first, then inspected.

cox_df = df[["time", "event", "CancerScore", "EpigeneticInstability"]].copy()

print("Cox model input columns:", cox_df.columns.tolist())
print(cox_df.describe())

cox = CoxPHFitter(penalizer=0.1)
cox.fit(cox_df, duration_col="time", event_col="event")
cox.print_summary()


## Step 16: Cox Model — Hazard Ratio Forest Plot

**Figure S15** (Supplementary Figure)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cox.plot(ax=ax)

ax.set_title("Figure S15. Cox Proportional Hazards Model\n"
             "Hazard Ratios for Epigenetic Features (Pseudo-Survival)",
             fontsize=11, fontweight="bold", pad=10)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("figures/FigS15_Cox_HazardRatios.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: figures/FigS15_Cox_HazardRatios.png")


## Summary of Results

In [ ]:
print("=" * 55)
print("  Meth3D-Net V6 — Pseudo-Clinical Risk Model Results")
print("=" * 55)
print(f"  Random Forest AUC (test):        {rf_auc:.4f}")
print(f"  XGBoost AUC (test):              {xgb_auc:.4f}")
print(f"  Cross-validation AUC (5-fold):   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()
print("  Saved figures:")
import glob
for f in sorted(glob.glob("figures/FigS*.png")):
    print(f"    {f}")
print("=" * 55)
